# Day 052 — Exercise 1: Your First Endpoint

**What you'll build:** `create_health_app()` — a FastAPI app with one route, `GET /health`, that returns `{'status': 'ok', 'model': 'llama3.2'}`. You'll test it with `TestClient` — no server needed.

**Why it matters:** Yesterday you built a UI. Today you build the other half of a real app: an HTTP API any client can call. FastAPI turns a Python function into a web endpoint with one decorator. And `TestClient` lets you call that endpoint *in-process* — the same way you'll test every API in this section, without ever starting a server.

## Provided: Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from starlette.testclient import TestClient
import ollama

## Your Implementation

In [ ]:
def create_health_app() -> FastAPI:
    """
    Return a FastAPI app with one route:
        GET /health -> {'status': 'ok', 'model': 'llama3.2'}
    """
    app = FastAPI(title='AI API', version='1.0.0')

    # TODO: define a GET /health route with a decorator:
    # @app.get('/health')
    # def health():
    #     return {'status': 'ok', 'model': 'llama3.2'}

    return app

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    # Check 1: create_health_app returns a FastAPI instance
    try:
        app = create_health_app()
        assert isinstance(app, FastAPI), f'expected FastAPI, got {type(app).__name__}'
        client = TestClient(app)
        passed += 1; print('✅ Check 1: create_health_app returns a FastAPI app')
    except Exception as e:
        print(f'❌ Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: GET /health returns HTTP 200
    try:
        r = client.get('/health')
        assert r.status_code == 200, f'expected 200, got {r.status_code}'
        passed += 1; print('✅ Check 2: GET /health -> 200')
    except Exception as e:
        print(f'❌ Check 2: {e}')

    # Check 3: body is JSON with status == 'ok'
    try:
        body = client.get('/health').json()
        assert body.get('status') == 'ok', f"expected status ok, got {body}"
        passed += 1; print('✅ Check 3: /health body has status == ok')
    except Exception as e:
        print(f'❌ Check 3: {e}')

    # Check 4: unknown route returns 404
    try:
        assert client.get('/does-not-exist').status_code == 404, 'expected 404'
        passed += 1; print('✅ Check 4: unknown route -> 404')
    except Exception as e:
        print(f'❌ Check 4: {e}')

    # Check 5: /health reports the model
    try:
        body = client.get('/health').json()
        assert 'model' in body, f"expected a model key, got {body}"
        passed += 1; print(f"✅ Check 5: /health reports model={body['model']!r}")
    except Exception as e:
        print(f'❌ Check 5: {e}')

    if passed == total:
        print('🎉 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def create_health_app() -> FastAPI:
    """Return a FastAPI app with a single liveness route.

    GET /health -> {'status': 'ok', 'model': 'llama3.2'}  (HTTP 200)
    """
    app = FastAPI(title='AI API', version='1.0.0')

    @app.get('/health')
    def health():
        return {'status': 'ok', 'model': 'llama3.2'}

    return app
```

**Why this works:** `FastAPI()` is the app object; `@app.get('/health')` registers the function below it as the handler for that path and method. Return a dict and FastAPI serialises it to JSON with a 200 status automatically. `TestClient(app)` wraps the app and lets you call it like an HTTP client (`client.get('/health')`) entirely in-process — no uvicorn, no ports, no network. That is how every endpoint in this section is tested.
</details>